## 1. Two paper findings + my methodology questions

In this section, I will practice reading the research paper like an ML engineer.

The goal is not to attack the paper. The goal is to ask careful methodology questions, the same way I should ask careful questions about my own model.

I will use respectful questions such as:

- Where does the label come from?
- What population does this result represent?
- Does the validation design support the claim?
- Could there be leakage?
- Does the result show causation, or only an observed pattern?

## 2. My model under an honest split

In Week 5, I used a grouped split by `content_hash_id`. That was better than a random row split because the same content page could appear on multiple dates.

For this validation audit, I will make the split stricter by grouping by `client_hash_id`.

This is more honest because it tests whether the model can generalize to unseen clients, not just unseen pages from clients it may have already seen during training.

In [1]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

In [3]:
!pip install -q huggingface_hub pyarrow

import os
import json
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import list_repo_files, hf_hub_download

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN is not set. Add HF_TOKEN in Colab Secrets first.")

repo_id = "FlyRank/internship-warehouse"
target_table = "fact_content_daily_performance"
target_month = "month=2026-03"

files = list_repo_files(
    repo_id,
    repo_type="dataset",
    token=HF_TOKEN
)

march_files = [
    f for f in files
    if target_table in f
    and target_month in f
    and f.endswith(".parquet")
]

print("March files found:", len(march_files))
print(march_files[:10])

March files found: 1
['fact_content_daily_performance/month=2026-03/data_0.parquet']


In [6]:
local_files = []

for file_path in march_files:
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=file_path,
        repo_type="dataset",
        token=HF_TOKEN
    )
    local_files.append(local_path)

df = pd.concat(
    [pd.read_parquet(path) for path in local_files],
    ignore_index=True
)

print("Rows loaded:", len(df))
print("Columns:", len(df.columns))

df.head()

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Rows loaded: 9841378
Columns: 30


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
work_df = df[df["gsc_data_available"].eq(True)].copy()
work_df = work_df[work_df["gsc_impressions"] > 0].copy()

work_df["gsc_ctr"] = np.where(
    work_df["gsc_impressions"] > 0,
    work_df["gsc_clicks"] / work_df["gsc_impressions"],
    0
)

model_df = work_df[work_df["ga4_sessions"] > 0].copy()

model_df["needs_review_proxy"] = (
    model_df["ga4_engaged_sessions"] == 0
).astype(int)

print("Rows for validation audit:", len(model_df))
print("Target distribution:")
print(model_df["needs_review_proxy"].value_counts())
print("Target rate:", round(model_df["needs_review_proxy"].mean(), 3))

Rows for validation audit: 361095
Target distribution:
needs_review_proxy
1    335986
0     25109
Name: count, dtype: int64
Target rate: 0.93


In [8]:
feature_cols = [
    "gsc_impressions",
    "gsc_avg_position",
    "gsc_ctr",
    "ga4_pageviews",
    "ga4_sessions",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_ai",
]

X = model_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = model_df["needs_review_proxy"]

In [9]:
groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

test_df = model_df.iloc[test_idx].copy()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train target rate:", round(y_train.mean(), 3))
print("Test target rate:", round(y_test.mean(), 3))
print("Train clients:", model_df.iloc[train_idx]["client_hash_id"].nunique())
print("Test clients:", model_df.iloc[test_idx]["client_hash_id"].nunique())

Train rows: 301641
Test rows: 59454
Train target rate: 0.93
Test target rate: 0.935
Train clients: 25
Test clients: 9


In [10]:
test_df["low_ctr_factor"] = 1 - test_df["gsc_ctr"].clip(0, 1)

test_df["position_factor"] = np.where(
    test_df["gsc_avg_position"] <= 20,
    1,
    0.5
)

test_df["baseline_score"] = (
    np.log1p(test_df["gsc_impressions"])
    * test_df["low_ctr_factor"]
    * test_df["position_factor"]
)

In [11]:
model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

test_df["model_score"] = model.predict_proba(X_test)[:, 1]

In [12]:
def precision_at_k(scores, y_true, k):
    result_df = pd.DataFrame({
        "score": scores,
        "target": y_true
    }).sort_values("score", ascending=False)

    return result_df.head(k)["target"].mean()


baseline_p50 = precision_at_k(
    test_df["baseline_score"],
    test_df["needs_review_proxy"],
    50
)

model_p50 = precision_at_k(
    test_df["model_score"],
    test_df["needs_review_proxy"],
    50
)

baseline_p100 = precision_at_k(
    test_df["baseline_score"],
    test_df["needs_review_proxy"],
    100
)

model_p100 = precision_at_k(
    test_df["model_score"],
    test_df["needs_review_proxy"],
    100
)

baseline_auc = roc_auc_score(
    test_df["needs_review_proxy"],
    test_df["baseline_score"]
)

model_auc = roc_auc_score(
    test_df["needs_review_proxy"],
    test_df["model_score"]
)

validation_comparison = pd.DataFrame({
    "method": ["Week 4 baseline", "Decision Tree model"],
    "precision_at_50": [baseline_p50, model_p50],
    "precision_at_100": [baseline_p100, model_p100],
    "roc_auc": [baseline_auc, model_auc],
    "split": ["grouped_by_client_hash_id", "grouped_by_client_hash_id"],
})

validation_comparison

,method,precision_at_50,precision_at_100,roc_auc,split
0,Week 4 baseline,0.68,0.65,0.380395,grouped_by_client_hash_id
1,Decision Tree model,1.00,1.00,0.868603,grouped_by_client_hash_id


## Honest split interpretation

For this validation audit, I used a stricter grouped split by `client_hash_id`.

This means the model was trained on some clients and tested on different clients. This is more honest than allowing the same client to appear in both train and test because pages from the same client may share similar traffic patterns, content structure, or audience behavior.

The comparison table measures the Week 4 baseline and the Decision Tree model on the same client-grouped test split.

This result should be interpreted as validation evidence for a proxy target, not proof that the model identifies true content refresh need. The output is still decision support and should be reviewed by a human.

## Client-grouped split interpretation

For this validation audit, I re-ran the Week 5 model using a stricter grouped split by `client_hash_id`.

This means the model was trained on some clients and tested on different clients. This is more honest than only grouping by `content_hash_id` because pages from the same client may share similar traffic patterns, audience behavior, site structure, or content strategy.

Under this stricter client-grouped split, the Week 4 baseline reached Precision@50 = 0.68, Precision@100 = 0.65, and ROC AUC = 0.380395.

The Decision Tree model reached Precision@50 = 1.00, Precision@100 = 1.00, and ROC AUC = 0.868603.

This measured result suggests that the Decision Tree ranked the proxy review candidates better than the baseline on this validation split.

However, this should still be treated carefully. The target is a proxy based on engagement behavior, not a direct label proving that a page truly needs content refresh. The result is useful as decision-support evidence, but a human should still review the page, query intent, content purpose, and business context before deciding what action to take.

In [14]:
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

feature_importance

,feature,importance
5,sessions_organic,0.737034
3,ga4_pageviews,0.126417
6,sessions_direct,0.048964
4,ga4_sessions,0.043776
7,sessions_referral,0.028015
2,gsc_ctr,0.007131
1,gsc_avg_position,0.006365
0,gsc_impressions,0.002298
8,sessions_social,0.000000
9,sessions_ai,0.000000


## Leakage audit

My target proxy is:

`needs_review_proxy = 1 if ga4_engaged_sessions == 0`

Because of that, I should not use `ga4_engaged_sessions` as an input feature. That would directly leak the answer into the model.

In this model, I excluded both `needs_review_proxy` and `ga4_engaged_sessions` from the feature list.

The feature importance table shows that the model relied mostly on `sessions_organic`, with importance 0.737034, and `ga4_pageviews`, with importance 0.126417. It also used smaller signals from `sessions_direct`, `ga4_sessions`, and `sessions_referral`.

This is not direct target leakage, but it still needs careful interpretation. The target is based on engagement behavior, and some of the important features come from related traffic and analytics signals. This means the model may be learning patterns in traffic and engagement, not true content refresh need.

Search-related features such as `gsc_ctr`, `gsc_avg_position`, and `gsc_impressions` had low importance in this model. This suggests the Decision Tree relied more on analytics behavior than search visibility.

Leakage decision:

- I excluded the target column from the model inputs.
- I excluded `ga4_engaged_sessions`, which directly defines the target proxy.
- I used a stricter client-grouped split by `client_hash_id`.
- I treated the target as a proxy, not ground truth.
- I will describe the output as decision support, not proof that a page needs refresh.

In [15]:
test_df["model_rank"] = test_df["model_score"].rank(
    method="first",
    ascending=False
)

test_df["model_top_100"] = test_df["model_rank"] <= 100

false_positives = test_df[
    (test_df["model_top_100"]) &
    (test_df["needs_review_proxy"] == 0)
].sort_values("model_score", ascending=False)

false_negatives = test_df[
    (~test_df["model_top_100"]) &
    (test_df["needs_review_proxy"] == 1)
].sort_values("model_score", ascending=True)

print("False positives in top 100:", len(false_positives))
print("False negatives outside top 100:", len(false_negatives))

False positives in top 100: 0
False negatives outside top 100: 55488


## Error examples interpretation

The model had 0 false positives in the top 100. This means every row in the model’s top 100 was labeled as `needs_review_proxy = 1`.

This looks strong for the top of the review queue. However, it does not mean the model is perfect. The model also had 55,488 false negatives outside the top 100. This means many rows were labeled as possible review candidates, but the model did not rank them in the top 100.

For a ranking task, this is expected because the model is only prioritizing the top review candidates. The goal is not to review every possible candidate immediately. The goal is to make the top of the review queue useful.

However, the high number of false negatives shows that the proxy target is broad. There are many rows that match the target definition, so the model should not be treated as a complete list of all pages that may need review.

This result should be described as decision support. A human reviewer still needs to inspect the page, search intent, content purpose, and business context before deciding what action to take.

In [16]:
false_positives[
    [
        "content_hash_id",
        "client_hash_id",
        "report_date",
        "model_score",
        "gsc_impressions",
        "gsc_ctr",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
        "needs_review_proxy",
    ]
].head(10)

,content_hash_id,client_hash_id,report_date,model_score,gsc_impressions,gsc_ctr,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,needs_review_proxy


In [17]:
false_negatives[
    [
        "content_hash_id",
        "client_hash_id",
        "report_date",
        "model_score",
        "gsc_impressions",
        "gsc_ctr",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
        "needs_review_proxy",
    ]
].head(10)

,content_hash_id,client_hash_id,report_date,model_score,gsc_impressions,gsc_ctr,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,needs_review_proxy
2340695,content_c44b6355a8e2b418,client_65de48885f4ef01b,2026-03-07,0.057057,99,0.0,4.474747,1.0,0.0,1
3770733,content_95eab5457121f730,client_fef1a8f436438636,2026-03-14,0.057057,8,0.0,0.000000,1.0,0.0,1
6788380,content_302b1534e040fe4c,client_fef1a8f436438636,2026-03-23,0.057057,18,0.0,43.111111,1.0,0.0,1
5559886,content_cbab33f725fe2de2,client_fef1a8f436438636,2026-03-18,0.057057,351,0.0,22.179487,1.0,0.0,1
3854362,content_1eb876da383f52c0,client_fef1a8f436438636,2026-03-12,0.057057,61,0.0,45.950820,1.0,0.0,1
7379772,content_cefc95c00835f8c8,client_fef1a8f436438636,2026-03-26,0.057057,8,0.0,16.500000,1.0,0.0,1
7380356,content_a37f9708f451d52f,client_fef1a8f436438636,2026-03-26,0.057057,178,0.0,5.528090,1.0,0.0,1
2636684,content_3e68e337445b33d4,client_fef1a8f436438636,2026-03-09,0.057057,3,0.0,3.666667,1.0,0.0,1
3040393,content_2635c4ea946cebcb,client_fef1a8f436438636,2026-03-11,0.057057,126,0.0,0.761905,1.0,0.0,1
6593091,content_2e9d8e1ea113a8e4,client_fef1a8f436438636,2026-03-21,0.057057,215,0.0,5.372093,1.0,0.0,1


## False negative examples

The false negative examples show rows that were labeled as `needs_review_proxy = 1`, but the model gave them a low score and did not rank them in the top 100.

Many of these rows have `gsc_ctr = 0.0`, meaning they had search impressions but no clicks. Some rows also have low or moderate impressions, such as 8, 18, 61, 99, 126, 178, 215, or 351 impressions.

This suggests that the model did not prioritize all zero-CTR rows equally. It may have learned that some low-impression or lower-signal rows are less urgent than higher-traffic candidates.

This is reasonable for a ranking system because the goal is not to flag every possible candidate at the top. The goal is to prioritize the strongest review candidates first.

However, these false negatives are still important. Some of these pages may deserve review, especially if they represent important content, valuable queries, or business-relevant pages. A human reviewer should still inspect lower-ranked candidates when time allows.

This supports a safer claim: the model helps prioritize review candidates, but it does not capture every page that may need review.

## 4. Claim rewrite

Original risky claim:
“The model identifies which pages need a content refresh.”

Safer claim:
“The model ranks pages that may deserve review based on observed search and engagement signals.”

Original risky claim:
“The Decision Tree proves which signals cause content decline.”

Safer claim:
“The Decision Tree measured which features helped separate the proxy target in this validation setup. This does not prove causation.”

Original risky claim:
“The model is correct because it beat the baseline.”

Safer claim:
“The model performed better than the Week 4 baseline on the measured proxy target and client-grouped validation split. The result should be treated as decision support and checked by a human reviewer.”

Original risky claim:
“The model finds all pages that need review.”

Safer claim:
“The model helps prioritize the top review candidates, but it can miss many other rows that match the proxy target.”

### Paper finding 1

Finding:
The paper reports that the study covers 341,701 content pieces across 57 brands, with 469,879,632 impressions, 1,514,819 clicks, 1,635,404 sessions, and 17,344 AI sessions.

Methodology question:
My question is: how were the 57 brands and 341,701 content pieces selected for the study?

This matters because the result depends on the population being measured. If the brands come from specific industries, traffic levels, client types, or data-availability requirements, the findings may not represent all websites or all brands. I would also ask whether every content piece had the same level of Google Search Console, GA4, and AI-session data available.

Careful interpretation:
This finding shows the measured scope of the dataset. I can describe it as a verified dataset summary for the paper, but I should not assume it represents the whole internet or all websites unless the paper explains that sampling design.

### Paper finding 2

Finding:
The paper uses several reporting windows: a rolling 90-day performance window, a 30-day trend comparison, a local historical monthly series, and local active-content feature-vector cuts where noted.

Methodology question:
My question is: do the reporting windows match the type of claim being made?

This matters because a 90-day performance window, a 30-day trend comparison, and a monthly historical series answer different questions. A 30-day trend may show a recent observed change, but it may not be enough to support a long-term claim. A 90-day window may smooth short-term noise, but it may also hide sudden changes. I would want each claim in the paper to clearly match the reporting window used to support it.

Careful interpretation:
This finding shows that the paper uses different measurement windows for different types of analysis. I should describe these results as measured within their stated reporting windows, not as universal or permanent patterns.

## 4. Claim rewrite

Original risky claim:
“The model identifies which pages need a content refresh.”

Safer claim:
“The model ranks pages that may deserve review based on observed search and engagement signals.”

Original risky claim:
“The Decision Tree proves which signals cause content decline.”

Safer claim:
“The Decision Tree measured which features helped separate the proxy target in this validation setup. This does not prove causation.”

Original risky claim:
“The model is correct because it beat the baseline.”

Safer claim:
“The model performed better than the Week 4 baseline on the measured proxy target and client-grouped validation split. The result should be treated as decision support and checked by a human reviewer.”

Original risky claim:
“The model finds all pages that need review.”

Safer claim:
“The model helps prioritize the top review candidates, but it can miss many other rows that match the proxy target.”

## 5. Self-check

- [x] I named two paper findings and asked methodology questions about each.
- [x] I framed the paper questions constructively.
- [x] I re-ran my Week 5 model under a stricter grouped split by `client_hash_id`.
- [x] I compared the Week 4 baseline and Decision Tree model on the same client-grouped split.
- [x] I reported Precision@50, Precision@100, and ROC AUC.
- [x] I audited features for leakage.
- [x] I excluded direct target leakage from the model inputs.
- [x] I looked at false positive and false negative examples.
- [x] I rewrote risky claims into safer claims.
- [x] I used careful language such as observed, measured, proxy, validation, and decision support.